# Medical Dataset Benchmarks

This notebook benchmarks Phil's representation-guided imputation against baseline methods on two open-source medical datasets:

1. **PIMA Indians Diabetes** (UCI ML Repository, id=34) — 768 patients, 8 numeric features
2. **Heart Disease Cleveland** (UCI ML Repository, id=45) — 303 patients, 13 mixed features

We evaluate four imputation strategies:

| Method | Description |
|--------|-------------|
| `SimpleImputer(mean)` | Column mean / most-frequent |
| `KNNImputer` | 5-nearest-neighbour average |
| `Phil(default)` | Ensemble of BayesianRidge / tree models, ECT representative selection |
| `Phil(covariate_sampling)` | k-NN conditional sampling + ECT representative selection |

**Metric:** mean absolute error (MAE) on cells that were artificially masked (MCAR, 15% rate).

---

### Installation

```bash
pip install philler[examples]
# or, from source:
uv sync --extra examples
```

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer, SimpleImputer
from ucimlrepo import fetch_ucirepo

from phil import Phil

warnings.filterwarnings("ignore")
rng = np.random.RandomState(42)

MASK_RATE = 0.15  # fraction of observed values to artificially remove

## Benchmarking utilities

In [ ]:
def introduce_mcar(df: pd.DataFrame, rate: float, seed: int = 42) -> tuple[pd.DataFrame, np.ndarray]:
    """Randomly mask *rate* fraction of observed numeric values (MCAR).

    Returns the masked DataFrame and a boolean mask array marking the
    positions that were removed (True = masked out).
    """
    rs = np.random.RandomState(seed)
    df_masked = df.copy()
    numeric_cols = df.select_dtypes(include="number").columns
    mask = np.zeros(df.shape, dtype=bool)

    for col in numeric_cols:
        observed = df[col].notna()
        observed_idx = np.where(observed)[0]
        n_mask = max(1, int(len(observed_idx) * rate))
        chosen = rs.choice(observed_idx, size=n_mask, replace=False)
        col_idx = df.columns.get_loc(col)
        df_masked.iloc[chosen, col_idx] = np.nan
        mask[chosen, col_idx] = True

    return df_masked, mask


def mae_on_mask(imputed: np.ndarray, truth: np.ndarray, mask: np.ndarray) -> float:
    """MAE restricted to masked positions."""
    return float(np.abs(imputed[mask] - truth[mask]).mean())


def run_benchmark(
    df_truth: pd.DataFrame,
    df_missing: pd.DataFrame,
    mask: np.ndarray,
    samples: int = 20,
    random_state: int = 42,
) -> dict[str, float]:
    """Run all four imputers and return MAE for each."""
    truth = df_truth.select_dtypes(include="number").values
    numeric_missing = df_missing.select_dtypes(include="number")
    numeric_mask = mask[:, [df_truth.columns.get_loc(c) for c in numeric_missing.columns]]

    results = {}

    # SimpleImputer (mean)
    si = SimpleImputer(strategy="mean")
    si_imputed = si.fit_transform(numeric_missing)
    results["SimpleImputer(mean)"] = mae_on_mask(si_imputed, truth, numeric_mask)

    # KNNImputer
    knn = KNNImputer(n_neighbors=5)
    knn_imputed = knn.fit_transform(numeric_missing)
    results["KNNImputer(k=5)"] = mae_on_mask(knn_imputed, truth, numeric_mask)

    # Phil default
    phil_default = Phil(samples=samples, param_grid="default", random_state=random_state)
    phil_df = phil_default.fit(df_missing)
    phil_numeric = phil_df[[c for c in phil_df.columns if c.startswith("num__")]].values
    results["Phil(default)"] = mae_on_mask(phil_numeric, truth, numeric_mask)

    # Phil covariate_sampling
    phil_cov = Phil(samples=samples, param_grid="covariate_sampling", random_state=random_state)
    phil_cov_df = phil_cov.fit(df_missing)
    phil_cov_numeric = phil_cov_df[[c for c in phil_cov_df.columns if c.startswith("num__")]].values
    results["Phil(covariate_sampling)"] = mae_on_mask(phil_cov_numeric, truth, numeric_mask)

    return results, phil_default, phil_cov


def print_results(results: dict[str, float], dataset_name: str):
    print(f"\n{'='*50}")
    print(f"  {dataset_name} — MAE on masked cells (lower is better)")
    print(f"{'='*50}")
    for method, mae in sorted(results.items(), key=lambda x: x[1]):
        print(f"  {method:<30s}  {mae:.4f}")
    print()

---
## Dataset 1: PIMA Indians Diabetes

The PIMA dataset encodes physiological missingness as zeros in biologically impossible columns (you cannot have zero glucose, BMI, etc.).  We replace those zeros with `NaN` to expose the true missing-value structure before benchmarking.

In [ ]:
pima_repo = fetch_ucirepo(id=34)  # Diabetes
pima_raw = pima_repo.data.features.copy()

# Biologically impossible zeros → NaN
zero_impossible_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
existing = [c for c in zero_impossible_cols if c in pima_raw.columns]
pima_raw[existing] = pima_raw[existing].replace(0, np.nan)

print("PIMA dataset shape:", pima_raw.shape)
print("\nMissing value counts:")
print(pima_raw.isnull().sum()[pima_raw.isnull().sum() > 0])

In [ ]:
# Keep only rows/cols that are fully observed (ground truth for MCAR evaluation)
pima_complete = pima_raw.dropna().reset_index(drop=True)
print(f"Complete rows for benchmarking: {len(pima_complete)}")

pima_masked, pima_mask = introduce_mcar(pima_complete, rate=MASK_RATE, seed=42)
print(f"Values masked: {pima_mask.sum()} / {pima_mask.size} ({100*pima_mask.mean():.1f}%)")

In [ ]:
pima_results, pima_phil_default, pima_phil_cov = run_benchmark(
    df_truth=pima_complete,
    df_missing=pima_masked,
    mask=pima_mask,
    samples=20,
    random_state=42,
)
print_results(pima_results, "PIMA Diabetes")

### MDS visualisation — PIMA (default grid)

Each point is one candidate imputation in ECT descriptor space.  The **red star** is the selected representative (closest to the centroid); the **blue diamond** is the centroid itself.

In [ ]:
fig, embedding = pima_phil_default.plot_mds(figsize=(9, 6), random_state=42)
fig.suptitle("PIMA — Phil(default) descriptor space", y=1.01)
plt.show()

print(f"Number of candidates: {len(pima_phil_default.magic_descriptors)}")
print(f"Selected candidate index: {pima_phil_default.closest_index}")

In [ ]:
fig, embedding = pima_phil_cov.plot_mds(figsize=(9, 6), random_state=42)
fig.suptitle("PIMA — Phil(covariate_sampling) descriptor space", y=1.01)
plt.show()

print(f"Number of candidates: {len(pima_phil_cov.magic_descriptors)}")
print(f"Selected candidate index: {pima_phil_cov.closest_index}")

---
## Dataset 2: Heart Disease (Cleveland)

The Cleveland Heart Disease dataset has genuine missing values in the `ca` (number of major vessels) and `thal` (thalassemia type) columns.  We use the complete rows as ground truth and introduce additional MCAR missingness for benchmarking.

In [ ]:
heart_repo = fetch_ucirepo(id=45)  # Heart Disease
heart_raw = heart_repo.data.features.copy()

print("Heart Disease dataset shape:", heart_raw.shape)
print("\nMissing value counts:")
print(heart_raw.isnull().sum()[heart_raw.isnull().sum() > 0])
heart_raw.head()

In [ ]:
heart_complete = heart_raw.dropna().reset_index(drop=True)
# Keep only numeric columns for MAE evaluation
heart_complete = heart_complete.select_dtypes(include="number").reset_index(drop=True)
print(f"Complete rows for benchmarking: {len(heart_complete)}")

heart_masked, heart_mask = introduce_mcar(heart_complete, rate=MASK_RATE, seed=42)
print(f"Values masked: {heart_mask.sum()} / {heart_mask.size} ({100*heart_mask.mean():.1f}%)")

In [ ]:
heart_results, heart_phil_default, heart_phil_cov = run_benchmark(
    df_truth=heart_complete,
    df_missing=heart_masked,
    mask=heart_mask,
    samples=20,
    random_state=42,
)
print_results(heart_results, "Heart Disease Cleveland")

### MDS visualisation — Heart Disease

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

heart_phil_default.plot_mds(ax=ax[0], random_state=42)
ax[0].set_title("Phil(default)")

heart_phil_cov.plot_mds(ax=ax[1], random_state=42)
ax[1].set_title("Phil(covariate_sampling)")

fig.suptitle("Heart Disease — ECT descriptor spaces", fontsize=14)
plt.tight_layout()
plt.show()

---
## Summary

In [ ]:
summary = pd.DataFrame(
    {
        "PIMA Diabetes (MAE)": pima_results,
        "Heart Disease (MAE)": heart_results,
    }
).sort_values("PIMA Diabetes (MAE)")

display(summary.style.highlight_min(color="lightgreen", axis=0).format("{:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (col, title) in zip(axes, [
    ("PIMA Diabetes (MAE)", "PIMA Indians Diabetes"),
    ("Heart Disease (MAE)", "Heart Disease Cleveland"),
]):
    vals = summary[col]
    colors = ["#f03e3e" if i == vals.idxmin() else "#adb5bd" for i in vals.index]
    ax.barh(vals.index, vals.values, color=colors)
    ax.set_xlabel("MAE (lower is better)")
    ax.set_title(title)
    ax.invert_yaxis()

fig.suptitle("Imputation MAE comparison", fontsize=14)
plt.tight_layout()
plt.show()

---

### Key takeaways

- **Phil(covariate_sampling)** conditions each imputed value on the *k* most similar observed rows rather than the marginal distribution, which often improves covariate alignment.
- The **MDS plot** makes it easy to inspect whether the candidate representations cluster tightly (good consensus) or spread widely (high imputation uncertainty).
- The **selected representative** (red star) sits near the centroid by construction — Phil never picks an outlier imputation.